# Homework 4 - Trading system

The three questions of Module 4: the CAGR of the best rule with a reduced feature set,
the probability threshold of the second best rule under a trade limit, and the number
of trades of the best predictor after the target is redefined.

**The outcome**

| # | Question | Answer |
|---|---|---|
| 1 | What is the CAGR of `pred10_rf_best_rule_55` in the new, reduced setup? | **1.136** (CAGR 1.135860) |
| 2 | What is the threshold of the second best rule, under five trades a day? | **0.56** (4.40 trades a day, CAGR 1.117720) |
| 3 | How many trades does the best predictor make with the redefined target? | **8** (8.01 a day) |

Three scenarios are run, one per cell, because each of them fits a forest on the full
panel and they do not fit in memory together. The results are merged into
`output/hw4_answers.json` by `homework4/src/merge_results.py`.

In [1]:
# Colab: uncomment and run the next line first.
# !pip install -q pandas numpy pyarrow scikit-learn yfinance curl_cffi
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def repo_root(start: Path) -> Path:
    """Walk up the tree until the repository layout is recognised."""
    for path in [start, *start.parents]:
        if (path / "requirements.txt").exists() and (path / "scripts").exists():
            return path
    return start


REPO = repo_root(Path.cwd().resolve())
HW = REPO / "homework4"
sys.path.insert(0, str(HW / "src"))

print("repository :", REPO.name)
print("python     :", sys.version.split()[0])

repository : repo
python     : 3.11.2


In [2]:
# The homework datasets are not in the repository. They are fetched from the course's
# Google Drive links by scripts/download_data.py, which is resumable and skips the
# files that are already there.
import subprocess

subprocess.run([sys.executable, str(REPO / "scripts" / "download_data.py")], check=False)

for name in ("stocks_df_combined_2026_09_18.parquet.brotli", "data_hw2.parquet"):
    path = REPO / "data" / name
    print(f"{'ok ' if path.exists() else 'MISSING'} {path.name}")

skipping stocks_df_combined_2026_09_18.parquet.brotli (already present)
skipping data_hw2.parquet (already present)
ok  stocks_df_combined_2026_09_18.parquet.brotli
ok  data_hw2.parquet


## The setup

The same temporal split as in Module 3, the same features, and the random forest of
the lecture. The simulation is the one of the module: the signals are counted per day,
the capital requirement is the size of one position times the number of days it is held
times the 75th percentile of the signals per day, and the CAGR is computed on that
capital.

One detail matters for the numbers: the reference notebook of Module 04 was produced
with scikit-learn 1.7.x. From 1.8.0 onwards the random forest changed, and the
simulation no longer reproduces the published table, so `requirements.txt` pins
`scikit-learn==1.7.2`.

In [3]:
import hw4_questions

print("scikit-learn:", __import__("sklearn").__version__)

scikit-learn: 1.7.2


## Scenario 1 - The baseline, 301 features

The reference setup: every dummy variable, the original target, and the random forest
with the rules of the lecture. This is the run that has to reproduce the published
numbers of the module.

In [4]:
baseline = hw4_questions.run_and_save("baseline")

### BASELINE (301 features): rows=202292 features=301 (NUMERICAL=186 DUMMIES=115)


                               count_investments  avg_investments_per_day  q75_investments_per_day  capital      CAGR
prediction                                                                                                           
pred0_manual_cci                             921                 1.875764                      2.0   6000.0  1.062799
pred1_manual_prev_g1                       19158                18.421154                     24.0  72000.0  1.118818
pred2_manual_prev_g1_and_snp               15244                21.715100                     25.0  75000.0  1.089718
pred3_manual_dgs10_5                       15910                31.947791                     33.0  99000.0  1.083696
pred4_manual_dgs10_fedfunds                15921                31.905812                     33.0  99000.0  1.036690
pred5_clf_10                                9861                10.261186                     16.0  48000.0  1.119541
pred6_clf_4                                 6677        

BASELINE pred10 CAGR: 1.093571344852016


Saved: homework4/output/hw4_baseline.json


## Scenario 2 - The reduced feature set, 208 features

The dummy variables are cut down to the ones that carry information, the forest is
refitted, and the rules are swept over a wider range of thresholds. This is the setup
that answers question 1, and the sweep is what answers question 2.

In [5]:
reduced = hw4_questions.run_and_save("reduced")

### REDUCED (208 features): rows=202292 features=208 (NUMERICAL=186 DUMMIES=22)


                               count_investments  avg_investments_per_day  q75_investments_per_day  capital      CAGR
prediction                                                                                                           
pred0_manual_cci                             921                 1.875764                     2.00   6000.0  1.062799
pred1_manual_prev_g1                       19158                18.421154                    24.00  72000.0  1.118818
pred2_manual_prev_g1_and_snp               15244                21.715100                    25.00  75000.0  1.089718
pred3_manual_dgs10_5                       15910                31.947791                    33.00  99000.0  1.083696
pred4_manual_dgs10_fedfunds                15921                31.905812                    33.00  99000.0  1.036690
pred5_clf_10                                9699                10.588428                    17.00  51000.0  1.120493
pred6_clf_4                                 6677        

REDUCED pred10 CAGR: 1.1358603966077971


Saved: homework4/output/hw4_reduced.json


## Scenario 3 - The redefined target

The target is redefined as "the price is more than 2.2% higher in thirty days", which
is a much rarer event, and the predictors are refitted on it. A rare target changes
what the tree splits on, and the number of trades of the best predictor changes with
it, which is what question 3 asks.

In [6]:
redefined = hw4_questions.run_and_save("redefined")

### REDEFINED TARGET (>1.022), 301 features): rows=202292 features=301 (NUMERICAL=186 DUMMIES=115)


                               count_investments  avg_investments_per_day  q75_investments_per_day  capital      CAGR
prediction                                                                                                           
pred0_manual_cci                             921                 1.875764                      2.0   6000.0  1.062799
pred1_manual_prev_g1                       19158                18.421154                     24.0  72000.0  1.118818
pred2_manual_prev_g1_and_snp               15244                21.715100                     25.0  75000.0  1.089718
pred3_manual_dgs10_5                       15910                31.947791                     33.0  99000.0  1.083696
pred4_manual_dgs10_fedfunds                15921                31.905812                     33.0  99000.0  1.036690
pred5_clf_10                                7337                 8.009825                     10.0  30000.0  1.121790
pred6_clf_4                                 7209        

Q3 pred5_clf_10: {'count_investments': 7337.0, 'gross_rev': 18975.555556737232, 'fees': -1467.400146484375, 'net_rev': 17508.155410252857, 'avg_net_rev_per_investment': 2.3862825964635217, 'avg_investments_per_day': 8.009825327510917, 'q75_investments_per_day': 10.0, 'capital': 30000.0, 'CAGR': 1.1217904241092584}


Saved: homework4/output/hw4_redefined.json


## The answers

In [7]:
# The three scenario files are merged into one answers file.
import json
import subprocess

subprocess.run(
    [sys.executable, str(HW / "src" / "merge_results.py")],
    check=True,
    cwd=str(REPO),
)

answers = json.loads((HW / "output" / "hw4_answers.json").read_text())
for key, value in answers.items():
    print("=" * 70)
    print(key.upper(), "-", value.get("question", ""))
    for field, item in value.items():
        if field == "question":
            continue
        if isinstance(item, (dict, list)) and len(str(item)) > 200:
            print("  " + field + ": <" + type(item).__name__ + " of " + str(len(item)) + ">")
        else:
            print("  " + field + ": " + str(item))

{
  "q1": {
    "question": "Find the CAGR for the pred10_rf_best_rule_55 using the new setup (with fewer dummy variables).",
    "n_features_original_setup": 301,
    "n_features_new_setup": 208,
    "cagr_original_setup": 1.093571344852016,
    "cagr_new_setup": 1.1358603966077971,
    "notebook_reference_original_cagr": 1.1195,
    "selected_option": 1.136,
    "best_predictor_new_setup": "pred10_rf_best_rule_55"
  },
  "q2": {
    "question": "What is the probability threshold for the second-best Random Forest decision rule, if you want to have the number of average trades per day BELOW 5?",
    "rf_rules": {
      "pred10_rf_best_rule_55": {
        "threshold": 0.55,
        "count_investments": 3514,
        "avg_investments_per_day": 5.1600587372,
        "q75_investments_per_day": 6.0,
        "capital": 18000.0,
        "CAGR": 1.1358603966,
        "under_5_trades_per_day": false
      },
      "pred11_rf_best_rule_65": {
        "threshold": 0.65,
        "count_investments